<table>
  <tr>
    <td><div align="left"><font size="30">Machine Vision Toolbox for Python — Interactive Demo</font></div></td>
    <td><img src="https://github.com/petercorke/machinevision-toolbox-python/raw/main/docs/figs/VisionToolboxLogo_NoBackgnd@2x.png" width="400" height="317" style="height:auto"></td>
  </tr>
</table>

A quick, runnable introduction to the toolbox. Works locally, on Google Colab, or entirely in your browser via [JupyterLite](https://jupyterlite.readthedocs.io) and [Pyodide](https://pyodide.org) -- no installation required either way. The cell below installs the toolbox the first time it is run (and will take a few seconds on Colab or in the browser).

## Configuring the Jupyter environment
We need to import some packages to help us with linear algebra (`numpy`), graphics (`matplotlib`), and machine vision (`machinevisiontoolbox`). If you're running locally you need to have these packages installed.

<div class="alert alert-info">
If you're running on CoLab or in the browser we have to first install machinevisiontoolbox, which is not preinstalled, so the next cell will be slow.
</div>

In [ ]:
# MVTB_BOOTSTRAP_CELL: installs the package, which takes a short time, so wait for the line "Running ..." below.  Generated by sync_bootstrap.py from _mvtb_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'machinevision-toolbox-python'
IMPORT_NAME = 'machinevisiontoolbox'
GIT_URL: str | None = 'https://github.com/petercorke/machinevision-toolbox-python'
JL_PACKAGES = ['numpy', 'scipy', 'matplotlib', 'ipympl', 'pillow', 'typing_extensions', 'opencv-python', 'ansitable', 'mvtb-data', 'tqdm', 'requests', 'ipywidgets', 'spatialmath-python', 'pgraph-python']
JL_NO_DEPS: list[str] = []
JL_PACKAGE_NO_DEPS = True
JL_EXTRAS: dict[str, list[str]] = {}
COLAB_PACKAGES: list[str] = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE, deps=not JL_PACKAGE_NO_DEPS)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed()


## Images and pixels

The core class is `Image`.  Let's read one of the bundled images and inspect it.

In [ ]:
%matplotlib ipympl
import matplotlib.pyplot as plt
plt.close("all")   # re-running the notebook from the top starts with no figures

from machinevisiontoolbox import Image

mona = Image.Read("monalisa.png")
print(f"size: {mona.width} x {mona.height}, dtype: {mona.dtype}, colour: {mona.iscolor}")


Display the image inline with `disp()`, which uses matplotlib.

In [ ]:
mona.disp()

## Smoothing

Apply a Gaussian blur and display original and result side-by-side.

In [ ]:
smooth = mona.smooth(sigma=3)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
mona.disp(ax=axes[0], title="Original")
smooth.disp(ax=axes[1], title="Smoothed (sigma=3)")
plt.tight_layout()

## Greyscale and histograms

In [ ]:
grey = mona.mono()
print(f"grey: {grey.width} x {grey.height}, planes: {grey.nplanes}")

hist, x = grey.hist()
fig, ax = plt.subplots()
plt.bar(x, hist, width=1, color="steelblue")
plt.xlabel("Pixel value")
plt.ylabel("Count")
plt.title("Greyscale histogram")
plt.tight_layout()

## Colour planes

Access individual colour planes by name — the toolbox tracks the colour order so
you never need to worry about BGR vs RGB.

In [ ]:
flowers = Image.Read("flowers1.png")
print(f"colour order: {flowers.colororder_str}")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, plane in zip(axes, ["R", "G", "B"]):
    flowers.plane(plane).disp(ax=ax, title=plane, colormap="gray")
plt.tight_layout()

## Binary blobs

Load a binary image and find the blobs.

In [ ]:
sharks = Image.Read("shark2.png")
blobs = sharks.blobs()
print(blobs)

fig, ax = plt.subplots()
sharks.disp(ax=ax)
blobs.plot_box(ax, color="g")
blobs.plot_centroid(ax, "o", color="y")
plt.tight_layout()

## Camera model

Create a central perspective camera and project a 3-D point into the image plane.

In [ ]:
from machinevisiontoolbox import CentralCamera
from spatialmath import SE3

cam = CentralCamera(f=0.015, rho=10e-6, imagesize=[1280, 1024],
                    pp=[640, 512], name="mycamera")
print(cam)

P = [0.3, 0.4, 3.0]
p = cam.project_point(P)
print(f"Projected pixel: {p}")

# Shift camera 100 mm to the right
p2 = cam.project_point(P, pose=SE3(0.1, 0, 0))
print(f"Projected pixel (camera shifted): {p2}")